In [ ]:
#STEP 0
import warnings
warnings.simplefilter("ignore")  # suppress all Python warnings

# suppress stdout/stderr messages in cells that print too much
import sys, os, contextlib
@contextlib.contextmanager
def suppress_output():
    with contextlib.redirect_stdout(open(os.devnull, "w")), \
         contextlib.redirect_stderr(open(os.devnull, "w")):
        yield

In [ ]:
#STEP 1
!pip install --quiet nltk gensim==4.3.2 pyLDAvis==3.4.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 22.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 93.4 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for gensim (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for gensim
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (gensim)


In [ ]:
#Step 2 (Could be merged with Step 1)
!pip install --force-reinstall "gensim==4.3.3" "scipy==1.11.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
#STEP 3 (Importing other required libraries)
# Ensure required packages are installed correctly
!pip install --quiet gensim pyLDAvis

#Ignore warnings====
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
#=======

import pandas as pd
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
from gensim.utils import simple_preprocess
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
import matplotlib.pyplot as plt

# Download necessary NLTK datasets
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)  # for lemmatization

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 74.2 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


True

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
#STEP 4 (Connecting with Google Drive and reading the file====)
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
#=======

from google.colab import drive
drive.mount('/content/drive')

# Replace 'your_file.csv' with the actual filename
file_path = '/content/drive/MyDrive/Colab Notebooks/DisneylandReviews.csv'
#file_path = '/content/drive/MyDrive/Colab Notebooks/Games1.csv'
df = pd.read_csv(file_path)

print(df.head())
print("Number of documents:", len(df))

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Mounted at /content/drive
                                         Review_Text
0  If you've ever been to Disneyland anywhere you...
1  Its been a while since d last time we visit HK...
2  Thanks God it wasn   t too hot or too humid wh...
3  HK Disneyland is a great compact park. Unfortu...
4  the location is not in the city, took around 1...
Number of documents: 42656


In [ ]:
#STEP 5 (Lemmatization)
#Ignore warnings====
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
#=======

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = simple_preprocess(str(text), deacc=True)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return tokens

#df['tokens'] = df['Content'].apply(preprocess)
df['tokens'] = df['Review_Text'].apply(preprocess)

In [ ]:
#STEP 6 (Bag of words creation)
#Ignore warnings====
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
#=======

dictionary = Dictionary(df['tokens'])
corpus = [dictionary.doc2bow(text) for text in df['tokens']]

print("Number of unique tokens:", len(dictionary))
print("Number of documents in corpus:", len(corpus))

Number of unique tokens: 35275
Number of documents in corpus: 42656


In [ ]:
#STEP 7 (Run loop for checking perplexity and coherence for different numbers of topics)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
#=======

import numpy as np
j=2
while j<9:
  lda_model = LdaModel(corpus=corpus,
                     id2word=dictionary,
                     num_topics=j,
                     update_every=1,
                     chunksize=50,
                     passes=10,
                     alpha='auto',
                     per_word_topics=True,
                     random_state=100)
  doc_lda = lda_model[corpus]
  coherence_model_lda = CoherenceModel(model=lda_model,
                                      texts=df['tokens'],   # these are your preprocessed token lists
                                      dictionary=dictionary,
                                      coherence='c_v')
  coherence_lda = coherence_model_lda.get_coherence()
  log_perplexity = lda_model.log_perplexity(corpus)
  perplexity = np.exp(-1 * log_perplexity)
  #print(f"Perplexity (positive): {perplexity}")
  print('\nNumber of topics = ',j, ": Perplexity (not log) =", perplexity, "Coherence Score = ", coherence_lda )
  #print('\nNumber of topics = ',j,  "Coherence Score = ", coherence_lda )
  j+=1


Number of topics =  2 : Perplexity (not log) = 1539.1314400730316 Coherence Score =  0.31601346832447463

Number of topics =  3 : Perplexity (not log) = 1557.5212949676559 Coherence Score =  0.3693301150208191

Number of topics =  4 : Perplexity (not log) = 1587.1828787519016 Coherence Score =  0.38423855387420247

Number of topics =  5 : Perplexity (not log) = 1630.105331485799 Coherence Score =  0.38952694658124154

Number of topics =  6 : Perplexity (not log) = 1690.731312293152 Coherence Score =  0.3818847806826305

Number of topics =  7 : Perplexity (not log) = 1753.5816783089433 Coherence Score =  0.3599878789024316

Number of topics =  8 : Perplexity (not log) = 1832.8618498632652 Coherence Score =  0.3574666053753511


In [ ]:
#STEP 8 (Locking the optimal number of topics)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
#========

num_topics = 3  # change as needed
#with suppress_output():
lda_model = LdaModel(corpus=corpus,
                       id2word=dictionary,
                       num_topics=num_topics,
                       random_state=42,
                       update_every=1,
                       chunksize=100,
                       passes=10,
                       alpha='auto',
                       per_word_topics=True)

# Correctly initialize the coherence model for the newly defined lda_model
coherence_model_lda = CoherenceModel(model=lda_model,
                                    texts=df['tokens'],   # your preprocessed token lists
                                    dictionary=dictionary,
                                    coherence='c_v')

coherence_lda = coherence_model_lda.get_coherence()
#print('\nNumber of topics = ',num_topics, ": Perplexity (not log) =", perplexity, "Coherence Score = ", coherence_lda )
print('\nNumber of topics = ',num_topics, "Coherence Score = ", coherence_lda )
# Print topics
#for idx, topic in lda_model.print_topics(-1):
#    print(f"Topic {idx}: {topic}\n")


Number of topics =  3 Coherence Score =  0.4149490368901469


In [ ]:
#STEP 9 (Creating visualization)

pyldavis_display = gensimvis.prepare(lda_model, corpus, dictionary)
pyLDAvis.display(pyldavis_display)

In [ ]:
#STEP 10 (Downloading visualization as html)
#Save and download
pyLDAvis.save_html(pyldavis_display, 'lda_visualization2025.html')

from google.colab import files
files.download('lda_visualization2025.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>